# Fase 1 · extração
LCEL, @chain, invoke, batch e stream. As células usam um **modelo duplo determinístico** e rodam sem credenciais. Uma célula, marcada como **execução real remota**, roda a mesma cadeia de ponta a ponta com o modelo real quando há chave no arquivo `.env`.

In [1]:
from pathlib import Path
import os
if Path.cwd().name == 'notebooks': os.chdir('..')

In [2]:
from langchain_core.runnables import RunnableLambda
from pedro.chains import criar_lcel, criar_chain
import json
modelo = RunnableLambda(lambda prompt: json.dumps({'intencao':'comparar',
    'solicitacao':{'produto_id':'PEC-01-0001','quantidade':2,'orcamento_centavos':100000,
    'prazo_max_dias':7},'faltantes':[]}, ensure_ascii=False))
lcel, decorada = criar_lcel(modelo), criar_chain(modelo)
entrada = {'texto':'Compare duas unidades de PEC-01-0001 até R$ 1.000 em 7 dias'}
print(lcel.invoke(entrada).model_dump_json())
print(decorada.invoke(entrada).model_dump_json())
print([x.model_dump() for x in lcel.batch([entrada, entrada])])
print(list(modelo.stream('etapa do modelo')))
print(lcel.get_graph().draw_mermaid())

{"intencao":"comparar","solicitacao":{"produto_id":"PEC-01-0001","produto":null,"quantidade":2,"orcamento_centavos":100000,"prazo_max_dias":7,"cep":null,"origem_preferida":null},"faltantes":[]}
{"intencao":"comparar","solicitacao":{"produto_id":"PEC-01-0001","produto":null,"quantidade":2,"orcamento_centavos":100000,"prazo_max_dias":7,"cep":null,"origem_preferida":null},"faltantes":[]}
[{'intencao': 'comparar', 'solicitacao': {'produto_id': 'PEC-01-0001', 'produto': None, 'quantidade': 2, 'orcamento_centavos': 100000, 'prazo_max_dias': 7, 'cep': None, 'origem_preferida': None}, 'faltantes': []}, {'intencao': 'comparar', 'solicitacao': {'produto_id': 'PEC-01-0001', 'produto': None, 'quantidade': 2, 'orcamento_centavos': 100000, 'prazo_max_dias': 7, 'cep': None, 'origem_preferida': None}, 'faltantes': []}]
['{"intencao": "comparar", "solicitacao": {"produto_id": "PEC-01-0001", "quantidade": 2, "orcamento_centavos": 100000, "prazo_max_dias": 7}, "faltantes": []}']
---
config:
  flowchart:


LCEL expõe etapas declarativas. @chain envolve a mesma extração. Os faltantes são recalculados depois do parser.

## A mesma cadeia de ponta a ponta com o modelo real
**Execução real remota** (OpenAI), só se houver um modelo configurado no arquivo `.env` (`OPENAI_API_KEY`, ou `PEDRO_BASE_URL` para um servidor local). O prompt estruturado, o modelo e o output parser de JSON rodam juntos, sem duplo, nos dois estilos de composição e com `invoke`, `batch` e `stream`. As solicitações são sintéticas. Sem modelo configurado, a célula avisa e não chama nada; a execução real salva está em `docs/rastros/fase1_chain_llm.json`.

In [3]:
from dotenv import load_dotenv
from pedro.chains import criar_lcel_parcial
from pedro.modelos import criar_llm, ha_modelo, nome_do_modelo
load_dotenv()
if ha_modelo():
    real = criar_llm(temperature=0)
    lcel_real, decorada_real = criar_lcel(real), criar_chain(real)
    pedidos = [{'texto': 'Compare 2 unidades de PEC-01-0001 até R$ 5.000 em 7 dias'},
               {'texto': 'Preciso de um termostato para geladeira, sem pressa'},
               {'texto': 'A que cidade corresponde o CEP 01001000?'}]
    print('modelo real:', nome_do_modelo())
    print('LCEL   invoke:', lcel_real.invoke(pedidos[0]).model_dump_json())
    print('@chain invoke:', decorada_real.invoke(pedidos[0]).model_dump_json())
    for pedido, saida in zip(pedidos, lcel_real.batch(pedidos)):
        print('batch :', pedido['texto'], '->', saida.intencao, '| faltam:', saida.faltantes)
    estados = list(criar_lcel_parcial(real).stream(pedidos[0]))
    print('stream:', len(estados), 'estados parciais; o último:', estados[-1])
else:
    print('Sem modelo configurado: esta célula não chama nada.')
    print('A execução real salva está em docs/rastros/fase1_chain_llm.json')

modelo real: gpt-4.1-mini


LCEL   invoke: {"intencao":"comparar","solicitacao":{"produto_id":"PEC-01-0001","produto":null,"quantidade":2,"orcamento_centavos":500000,"prazo_max_dias":7,"cep":null,"origem_preferida":null},"faltantes":[]}


@chain invoke: {"intencao":"comparar","solicitacao":{"produto_id":"PEC-01-0001","produto":null,"quantidade":2,"orcamento_centavos":500000,"prazo_max_dias":7,"cep":null,"origem_preferida":null},"faltantes":[]}


batch : Compare 2 unidades de PEC-01-0001 até R$ 5.000 em 7 dias -> comparar | faltam: []
batch : Preciso de um termostato para geladeira, sem pressa -> outro | faltam: ['quantidade']
batch : A que cidade corresponde o CEP 01001000? -> cep | faltam: ['produto', 'quantidade']


stream: 20 estados parciais; o último: {'intencao': 'comparar', 'solicitacao': {'produto_id': 'PEC-01-0001', 'produto': None, 'quantidade': 2, 'orcamento_centavos': 500000, 'prazo_max_dias': 7, 'cep': None, 'origem_preferida': None}, 'faltantes': []}


## R03 · modelo de prompt reutilizável com variáveis do domínio
Execução real local, sem modelo: o mesmo `ChatPromptTemplate` é preenchido com três solicitações diferentes e as mensagens resultantes são mostradas. O modelo de prompt fica em `pedro.chains.PROMPT` e é reutilizado pela cadeia LCEL e pela cadeia `@chain`. A execução com o modelo real está em `docs/rastros/fase1_chain_llm.json`.

In [4]:
from pedro.chains import PROMPT
print('classe:', type(PROMPT).__name__, '| variáveis:', sorted(PROMPT.input_variables))
print('mensagens:', [type(m).__name__ for m in PROMPT.messages])
parcial = PROMPT.partial(formato='Devolva JSON com intencao, solicitacao e faltantes.')
print('depois do partial -> variáveis livres:', sorted(parcial.input_variables))
for texto in ('Compare 2 unidades de PEC-01-0001 até R$ 5.000 em 7 dias',
              'Preciso de um termostato para geladeira, sem pressa',
              'A qual cidade corresponde o CEP 01001000?'):
    sistema, humano = parcial.format_messages(texto=texto)
    print(humano.type, '->', humano.content)
print(sistema.type, '->', sistema.content)
print('mesmo modelo de prompt na LCEL:', lcel.first.messages == PROMPT.messages)

classe: ChatPromptTemplate | variáveis: ['formato', 'texto']
mensagens: ['SystemMessagePromptTemplate', 'HumanMessagePromptTemplate']
depois do partial -> variáveis livres: ['texto']
human -> Compare 2 unidades de PEC-01-0001 até R$ 5.000 em 7 dias
human -> Preciso de um termostato para geladeira, sem pressa
human -> A qual cidade corresponde o CEP 01001000?
system -> Persona: Pedro. Tarefa: extrair intenção e solicitação de compra fictícia. Contexto: nenhum número ausente é inferido como zero. Formato: JSON válido. Devolva JSON com intencao, solicitacao e faltantes.
mesmo modelo de prompt na LCEL: True


## R04 · laboratório de saídas quebradas
Oito saídas típicas de um modelo passam pela cadeia com reparo (`criar_lcel_robusta`). Cada defeito é resolvido por uma camada diferente: o **parser** (tolera cercas e texto ao redor), o **reparo** (outra cadeia corrige com a solicitação, a saída inválida e o erro) ou uma **regra de domínio** (os faltantes são calculados pelo código). **Duplo de teste:** o gerador emite a saída quebrada e o reparador devolve sempre o JSON correto; o reparo com o modelo real está em `docs/rastros/fase1_chain_llm.json`.

In [5]:
from pedro.laboratorio import CORRETA, SAIDAS_QUEBRADAS, SOLICITACAO, ensaiar
reparador_duplo = RunnableLambda(lambda prompt: json.dumps(CORRETA))
print('solicitação:', SOLICITACAO)
for caso in ensaiar(reparador_duplo):
    print(f"{caso['caso']:28} camada: {caso['camada_observada']:17} reparos: {caso['reparos']}"
          f" -> {caso['resultado']['solicitacao']} faltam {caso['resultado']['faltantes']}")

solicitação: Compare 2 unidades de PEC-01-0001 até R$ 5.000 em 7 dias
JSON válido                  camada: nenhuma           reparos: 0 -> {'produto_id': 'PEC-01-0001', 'quantidade': 2, 'orcamento_centavos': 500000, 'prazo_max_dias': 7} faltam []
cercas markdown e conversa   camada: parser            reparos: 0 -> {'produto_id': 'PEC-01-0001', 'quantidade': 2, 'orcamento_centavos': 500000, 'prazo_max_dias': 7} faltam []
JSON truncado                camada: reparo            reparos: 1 -> {'produto_id': 'PEC-01-0001', 'quantidade': 2, 'orcamento_centavos': 500000, 'prazo_max_dias': 7} faltam []
quantidade por extenso       camada: reparo            reparos: 1 -> {'produto_id': 'PEC-01-0001', 'quantidade': 2, 'orcamento_centavos': 500000, 'prazo_max_dias': 7} faltam []
quantidade negativa          camada: reparo            reparos: 1 -> {'produto_id': 'PEC-01-0001', 'quantidade': 2, 'orcamento_centavos': 500000, 'prazo_max_dias': 7} faltam []
intenção fora da lista       camada: reparo  

## R04 · um parser tolerante pode perder dados em silêncio
Achado do laboratório. O parser do LangChain aceita JSON cortado. Com a cadeia simples, uma saída truncada vira um objeto válido ao qual falta o prazo, sem nenhum erro. A cadeia robusta confere que o JSON esteja inteiro e, se não estiver, manda reparar.

In [6]:
truncada = SAIDAS_QUEBRADAS['JSON truncado'][0]
print('saída do modelo:', truncada)
simples = criar_lcel(RunnableLambda(lambda _: truncada)).invoke({'texto': SOLICITACAO})
print('cadeia simples  -> prazo:', simples.solicitacao.prazo_max_dias, '(perdeu-se sem erro)')
from pedro.chains import criar_lcel_robusta
robusta = criar_lcel_robusta(RunnableLambda(lambda _: truncada), reparador_duplo).invoke({'texto': SOLICITACAO})
print('cadeia robusta -> prazo:', robusta.solicitacao.prazo_max_dias, '(detectado e reparado)')

saída do modelo: {"intencao": "comparar", "faltantes": [], "solicitacao": {"produto_id": "PEC-01-0001", "quantidade": 2, "orcamento_centavos": 500000, "prazo_ma
cadeia simples  -> prazo: None (perdeu-se sem erro)
cadeia robusta -> prazo: 7 (detectado e reparado)


## R05 · invoke, batch e stream sobre uma cadeia @chain
**Duplo de teste:** o modelo é um gerador que entrega o JSON em pedaços de 12 caracteres, para que `stream()` tenha algo a transmitir. Os três métodos são executados sobre a cadeia imperativa. `stream()` só transmite se todos os passos permitirem: com o parser Pydantic chega um único fragmento; com o parser de JSON parcial o objeto chega à medida que se forma. As medições com o modelo real estão em `docs/rastros/fase1_chain_llm.json`.

In [7]:
from pedro.chains import criar_chain_robusta, criar_lcel_parcial
texto_json = json.dumps(CORRETA)
def modelo_em_pedacos(_):
    for i in range(0, len(texto_json), 12):
        yield texto_json[i:i + 12]
duplo = RunnableLambda(modelo_em_pedacos)
imperativa = criar_chain(duplo)
entrada = {'texto': SOLICITACAO}
print('invoke:', imperativa.invoke(entrada).solicitacao.model_dump(exclude_none=True))
print('batch :', [r.intencao for r in imperativa.batch([entrada] * 3)])
print('stream com @chain e Pydantic:', len(list(imperativa.stream(entrada))), 'fragmento')
estados = list(criar_lcel_parcial(duplo).stream(entrada))
print('stream com JSON parcial:', len(estados), 'estados')
for estado in estados[::3]:
    print('  ', estado)

invoke: {'produto_id': 'PEC-01-0001', 'quantidade': 2, 'orcamento_centavos': 500000, 'prazo_max_dias': 7}
batch : ['comparar', 'comparar', 'comparar']
stream com @chain e Pydantic: 1 fragmento
stream com JSON parcial: 8 estados
   {}
   {'intencao': 'comparar', 'faltantes': [], 'solicitacao': {}}
   {'intencao': 'comparar', 'faltantes': [], 'solicitacao': {'produto_id': 'PEC-01-0001', 'quantidade': 2, 'orcamento_centavos': 500000}}


## R05 · a cadeia com reparo, escrita com @chain
`criar_chain_robusta` faz o mesmo que a versão declarativa de R04, mas o controle de erros é um `try/except` à vista. O mesmo laboratório passa pelas duas versões.

In [8]:
import inspect
print(inspect.getsource(criar_chain_robusta).split('    @chain\n')[1].split('    return extrair_com_reparo')[0])
print('imperativa = declarativa nos 8 casos:',
      ensaiar(reparador_duplo, criar_chain_robusta) == ensaiar(reparador_duplo))

    def extrair_com_reparo(dado: dict) -> Extracao:
        saida = gerar.invoke(dado)
        try:
            extracao = parser.parse(saida)
            if not _json_completo(saida):
                raise OutputParserException("JSON incompleto: a saída foi cortada antes de fechar")
        except OutputParserException as erro:
            corrigida = reparar.invoke({"texto": dado["texto"], "saida": saida, "erro": erro})
            extracao = parser.parse(corrigida)
        return _validar(extracao)


imperativa = declarativa nos 8 casos: True


## R06 · a mesma cadeia em LCEL e em @chain: o que cada uma deixa ver
Sem modelo (duplos). As duas versões da cadeia com reparo dão o mesmo resultado. Aqui se mede o que muda: os passos que o rastro registra, se o erro fica anotado e se o grafo mostra o reparo.

In [9]:
from pedro.laboratorio import comparar_estilos
medidas = comparar_estilos()
for estilo, dado in medidas.items():
    print(estilo)
    for chave, valor in dado.items():
        if chave != 'passos_com_reparo':
            print('  ', chave, '=', valor)
    print('   passos:', ' > '.join(dado['passos_com_reparo']))
print(criar_lcel_robusta(RunnableLambda(lambda _: ''), reparador_duplo).get_graph().draw_mermaid())

declarativa (LCEL)
   passos_rastreados_sem_falha = 10
   passos_rastreados_com_reparo = 16
   o_erro_fica_no_rastro = True
   a_validacao_de_dominio_fica_no_rastro = True
   o_grafo_mostra_o_reparo = True
   resultado_apos_reparar = 2
   passos: RunnableSequence > RunnableAssign<saida> > RunnableParallel<saida> > RunnableSequence > ChatPromptTemplate > RunnableLambda > StrOutputParser > RunnableWithFallbacks > analisar_estrito > ERRO:OutputParserException > RunnableSequence > ChatPromptTemplate > RunnableLambda > StrOutputParser > PydanticOutputParser > _validar
imperativa (@chain)
   passos_rastreados_sem_falha = 5
   passos_rastreados_com_reparo = 9
   o_erro_fica_no_rastro = False
   a_validacao_de_dominio_fica_no_rastro = False
   o_grafo_mostra_o_reparo = False
   resultado_apos_reparar = 2
   passos: extrair_com_reparo > RunnableSequence > ChatPromptTemplate > RunnableLambda > StrOutputParser > RunnableSequence > ChatPromptTemplate > RunnableLambda > StrOutputParser
---
config:


Leitura: na LCEL o rastro registra a falha do parser, o reparo e a validação de domínio como passos com nome, e o grafo mostra o `WithFallbacks`. Em `@chain` o resultado é idêntico, mas o `try/except` e a chamada a `_validar` são Python normal: o rastro não os vê.

## R01 · anatomia do agente
A anatomia é lida dos objetos reais do agente final (`pedro.anatomia`), não é escrita à mão. Nenhum modelo é chamado: o executor é construído com um modelo de preenchimento que nunca é invocado.

In [10]:
import json
from pedro.catalog import criar_cenario
from pedro.anatomia import descrever_anatomia
cenario = criar_cenario(Path('storage/notebooks/catalogos'))
memoria = Path('storage/notebooks/r01.sqlite3')
anatomia = descrever_anatomia(cenario, memoria)
for chave in ('motor_cognitivo', 'orquestrador', 'system_message', 'memoria',
              'base_conhecimento', 'porta_publicacao'):
    print(chave, json.dumps(anatomia[chave], ensure_ascii=False))
for f in anatomia['ferramentas']:
    print('ferramenta', f['nome'], '|', f['papel'], '|', f['logica'], '|', f['argumentos'])

motor_cognitivo {"classe": "ChatOpenAI", "modelo": "gpt-4.1-mini", "disponivel": true}
orquestrador {"agente": "create_openai_tools_agent", "executor": "AgentExecutor", "max_iteracoes": 6, "historico": "RunnableWithMessageHistory"}
system_message {"ptcf": {"Persona": true, "Tarefa": true, "Contexto": true, "Formato": true}, "variaveis": ["agent_scratchpad", "input"], "variaveis_fixadas": ["chat_history", "dataset_id", "exemplos", "formato"], "mensagens": ["SystemMessagePromptTemplate", "MessagesPlaceholder", "HumanMessagePromptTemplate", "MessagesPlaceholder"]}
memoria {"curto_prazo": "janela de 12 mensagens + rascunho por sessão", "longo_prazo": "preferências explícitas + episódios vetoriais por usuário", "modelo_embeddings": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"}
base_conhecimento {"dataset_id": "bdf1585e-7e71-4ea9-94e2-bb4b4d27fb4b", "pecas": 4320, "fragmentos_politicas": 75, "origem_politicas": "ficticio_demo"}
porta_publicacao {"verificacao": "exata (total,

## R01 · agente híbrido: camada reativa e camada deliberativa
Três turnos sobre o mesmo fluxo final. **Duplo de teste:** o modelo é um roteiro determinístico (não é OpenAI) e o ViaCEP está simulado. O real é o fluxo: as regras, o `AgentExecutor`, a ferramenta executada e a verificação antes de publicar. Os dois primeiros turnos são resolvidos por uma regra sem passar pelo agente; no segundo, o intérprete de turno lê a mensagem e a regra pergunta a quantidade. O terceiro passa pelo agente, que recebe as seis ferramentas e solicita uma.

In [11]:
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda
from pedro.agent import Rastro, criar_fluxo
from pedro.memory import nova_sessao
sessao = nova_sessao(memoria, 'aluna', cenario.dataset_id)
final = json.dumps({'estado': 'cep', 'dataset_id': cenario.dataset_id,
    'solicitacao': {'cep': '01001000'}, 'cidade': 'Cidade Simulada', 'uf': 'SP',
    'modo_cep': 'simulado', 'evidencias': [{'tipo': 'viacep',
    'fonte_id': 'ViaCEP:01001000', 'texto': 'Cidade Simulada/SP'}]})
roteiro = [AIMessage(content='', tool_calls=[{'name': 'consultar_cep',
                                               'args': {'cep': '01001000'}, 'id': 'c1'}]),
           AIMessage(content=final)]
chamadas, leituras = [], []
def modelo_duplo(prompt, **kwargs):
    if 'tools' not in kwargs:  # quem chama é o intérprete de turno, não o agente
        leituras.append(1)
        return json.dumps({'e_compra': 'CEP' not in prompt.to_messages()[-1].content})
    chamadas.append(sorted(t['function']['name'] for t in kwargs['tools']))
    return roteiro[len(chamadas) - 1]
rastro = Rastro()
fluxo = criar_fluxo(cenario, memoria, llm=RunnableLambda(modelo_duplo),
                    cep_simulado=True, rastro=rastro)
config = {'configurable': {'user_id': 'aluna', 'session_id': sessao}}
for pergunta in ('lista peças', 'Compare PEC-01-0001',
                 'A qual cidade corresponde o CEP 01001000?'):
    antes = (len(chamadas), len(rastro.eventos), len(leituras))
    resposta = fluxo.invoke({'input': pergunta}, config=config).content
    usadas = [e['ferramenta'] for e in rastro.eventos[antes[1]:] if e['evento'] == 'tool_start']
    n = len(chamadas) - antes[0]
    print('turno:', pergunta)
    print('  camada:', 'deliberativa' if n else 'reativa', '| chamadas ao agente:', n,
          '| leituras do intérprete:', len(leituras) - antes[2], '| ferramentas executadas:', usadas)
    print('  resposta:', resposta[:100].replace('\n', ' '))
print('ferramentas oferecidas ao modelo:', chamadas[0])
assert len(chamadas) == 2 and [e['ferramenta'] for e in rastro.eventos
                               if e['evento'] == 'tool_start'] == ['consultar_cep']

turno: lista peças
  camada: reativa | chamadas ao agente: 0 | leituras do intérprete: 0 | ferramentas executadas: []
  resposta: Peças do cenário (4320; página 1): - PEC-01-0001 · GELA-COMP-0001 · Compressor para Geladeiras GELA-
turno: Compare PEC-01-0001
  camada: reativa | chamadas ao agente: 0 | leituras do intérprete: 1 | ferramentas executadas: []
  resposta: De quantas unidades de PEC-01-0001 você precisa? Posso comparar sem orçamento ou prazo máximo se voc
turno: A qual cidade corresponde o CEP 01001000?
  camada: deliberativa | chamadas ao agente: 2 | leituras do intérprete: 1 | ferramentas executadas: ['consultar_cep']
  resposta: CEP 01001000: Cidade Simulada/SP (ViaCEP, modo simulado).
ferramentas oferecidas ao modelo: ['buscar_produtos', 'comparar_ofertas', 'consultar_catalogo', 'consultar_cep', 'consultar_politicas', 'salvar_preferencia']


/tmp/ipykernel_3529239/2683778477.py:21: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  fluxo = criar_fluxo(cenario, memoria, llm=RunnableLambda(modelo_duplo),


Leitura: «lista peças» e «Compare PEC-01-0001» são resolvidas por regra (0 chamadas ao agente); na segunda o intérprete confirma que é uma compra. A pergunta de CEP não tem regra: o modelo escolhe `consultar_cep` entre seis ferramentas e a resposta só é publicada depois de coincidir com a observação da ferramenta. Com um modelo real a escolha é feita pelo LLM; essa execução remota corresponde a R26.

## R02 · planejamento: uma solicitação complexa dividida em passos
Execução real local, sem modelo. A solicitação é dividida em cinco subtarefas, cada uma com uma saída observável: extrair restrições, consultar ofertas, comparar, verificar e publicar.

In [12]:
from pedro.agent import renderizar
from pedro.business import comparar
from pedro.catalog import ofertas
from pedro.dialogue import interpretar_compra
from pedro.verification import resposta_local, verificar
mensagem = 'Preciso de 5 unidades de PEC-01-0001 até R$ 10.000 em 7 dias'
estado, esclarecimento = interpretar_compra(cenario, None, mensagem)
print('1 extrair  ', estado.solicitacao.model_dump(exclude_none=True), '| falta:', esclarecimento)
disponiveis = ofertas(cenario, estado.solicitacao.produto_id)
print('2 consultar', [o.id for o in disponiveis])
alternativas = comparar(estado.solicitacao, disponiveis)
print('3 comparar ', [(a.oferta.fornecedor_id, a.total_centavos, a.atende, a.motivos)
                      for a in alternativas])
proposta = resposta_local(cenario, estado.solicitacao)
print('4 verificar', 'erros:', verificar(proposta, cenario))
print('5 publicar ', renderizar(proposta).splitlines()[0])

1 extrair   {'produto_id': 'PEC-01-0001', 'quantidade': 5, 'orcamento_centavos': 1000000, 'prazo_max_dias': 7} | falta: None
2 consultar ['PEC-01-0001-BR-05', 'PEC-01-0001-BR-06', 'PEC-01-0001-INT-06']
3 comparar  [('BR-06', 727983, True, []), ('BR-05', 193040, False, ['estoque', 'prazo']), ('INT-06', 734492, False, ['prazo'])]
4 verificar erros: []
5 publicar  CentroPeças: R$ 7.279,83 para 5 unidades; prazo de 2 dias.


## R02 · memória aumentada: o que vai para o curto e para o longo prazo
Execução real local, sem modelo. Curto prazo: o rascunho de compra e a janela de 12 mensagens vivem na sessão. Longo prazo: a preferência vive no usuário e aparece em uma sessão nova; outro usuário não a vê. Os episódios vetoriais são demonstrados no notebook 03.

In [13]:
from pedro.memory import Historico, salvar_preferencia, ler_rascunho, preferencias
sessao_a = nova_sessao(memoria, 'aluna', cenario.dataset_id)
local = criar_fluxo(cenario, memoria)
def cfg(usuario, sessao):
    return {'configurable': {'user_id': usuario, 'session_id': sessao}}
for texto in ('Preciso de 5 unidades de PEC-01-0001 em 7 dias', 'agora 6 unidades'):
    saida = local.invoke({'input': texto}, config=cfg('aluna', sessao_a)).content
    print(texto, '->', saida.splitlines()[0])
    print('  rascunho (curto prazo):', {k: v for k, v in
          ler_rascunho(memoria, 'aluna', sessao_a)['solicitacao'].items() if v is not None})
for _ in range(7):
    local.invoke({'input': 'lista peças'}, config=cfg('aluna', sessao_a))
historico = Historico(memoria, 'aluna', sessao_a)
print('mensagens salvas:', len(historico.todos()), '| janela que o agente vê:', len(historico.messages))
salvar_preferencia(memoria, 'aluna', 'fornecedor', 'nacional', explicita=True)
sessao_b = nova_sessao(memoria, 'aluna', cenario.dataset_id)
print('sessão nova -> rascunho:', ler_rascunho(memoria, 'aluna', sessao_b),
      '| preferência (longo prazo):', preferencias(memoria, 'aluna'),
      '| outro usuário:', preferencias(memoria, 'outro'))

Preciso de 5 unidades de PEC-01-0001 em 7 dias -> CentroPeças: R$ 7.279,83 para 5 unidades; prazo de 2 dias.
  rascunho (curto prazo): {'produto_id': 'PEC-01-0001', 'quantidade': 5, 'prazo_max_dias': 7}
agora 6 unidades -> CentroPeças: R$ 8.718,75 para 6 unidades; prazo de 2 dias.
  rascunho (curto prazo): {'produto_id': 'PEC-01-0001', 'quantidade': 6, 'prazo_max_dias': 7}


/tmp/ipykernel_3529239/2628486959.py:3: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  local = criar_fluxo(cenario, memoria)


mensagens salvas: 18 | janela que o agente vê: 12
sessão nova -> rascunho: None | preferência (longo prazo): {'fornecedor': 'nacional'} | outro usuário: {}


Leitura: ao mudar a quantidade para 6, o prazo de 7 dias é conservado no rascunho. O histórico guarda 18 mensagens, mas o agente só vê 12. A sessão nova começa sem rascunho e conserva a preferência do usuário.